In [ ]:
# Example 1: Process — spawn a child process running a target function.
import billiard

def worker(n, out_queue):
    out_queue.put(n * n)

q = billiard.Queue()
p = billiard.Process(target=worker, args=(7, q))
p.start()
p.join(timeout=10)
result = q.get(timeout=5)
print(f"worker returned: {result}")
assert result == 49
assert p.exitcode == 0


In [ ]:
# Example 2: Pool — parallel map over a function.
import billiard

def square(x):
    return x * x

with billiard.Pool(processes=2) as pool:
    results = pool.map(square, range(6))

print(f"pool.map results: {results}")
assert results == [0, 1, 4, 9, 16, 25]


In [ ]:
# Example 3: Pipe — bidirectional communication between processes.
import billiard

def echo(conn):
    msg = conn.recv()
    conn.send(f"echo: {msg}")
    conn.close()

parent_conn, child_conn = billiard.Pipe()
p = billiard.Process(target=echo, args=(child_conn,))
p.start()
parent_conn.send("hello")
reply = parent_conn.recv()
p.join(timeout=10)
print(f"reply from child: {reply}")
assert reply == "echo: hello"


In [ ]:
# Example 4: Lock + Value — shared synchronization primitives.
import billiard

def increment(counter, lock, n):
    for _ in range(n):
        with lock:
            counter.value += 1

counter = billiard.Value('i', 0)
lock = billiard.Lock()
procs = [billiard.Process(target=increment, args=(counter, lock, 50)) for _ in range(3)]
for p in procs:
    p.start()
for p in procs:
    p.join(timeout=10)

print(f"final counter: {counter.value}")
assert counter.value == 150


In [ ]:
# Example 5: get_context — cpu_count + explicit start-method introspection.
import billiard

ncpu = billiard.cpu_count()
print(f"cpu_count = {ncpu}")
assert ncpu >= 1

# billiard.get_context('spawn') must report the requested start method. We do NOT
# run a Pool under 'spawn' inside the notebook: a spawned child re-imports
# __main__ (the interactive kernel), which cannot resolve a cell-defined worker
# and hangs the cell. Verify the spawn context's start method, then exercise a
# real context-based Pool via 'fork' (the Linux default) using a picklable
# builtin so no cell-defined symbol needs to cross the process boundary.
spawn_ctx = billiard.get_context('spawn')
print(f"spawn context start method: {spawn_ctx.get_start_method()}")
assert spawn_ctx.get_start_method() == 'spawn'

fork_ctx = billiard.get_context('fork')
assert fork_ctx.get_start_method() == 'fork'
with fork_ctx.Pool(processes=2) as pool:
    results = pool.map(abs, [-10, -20, -30])

print(f"fork-context pool results: {results}")
assert results == [10, 20, 30]